# SECEP teacher — Phase B (Kaggle 2×T4, vLLM)

**This notebook, part 1 = setup + throughput go/no-go** (0–1.5 h box). Parts 2–3 (B1 prompt tuning, B2 pseudo-labelling) get appended below.

### Before you press Run
1. **Settings → Accelerator: GPU T4 ×2.** Internet **On** (needs a phone-verified Kaggle account). Persistence: Files only.
2. **Add Input → your dataset `secep-teacher-inputs`**, containing only:
   - `gold_dev.*`, `gold_calib.*` (from `Data/gold_final/`, csv/parquet/xlsx all fine)
   - `labelling-guide.md` (v2), `human_baseline.json`
   - `common/text.py`
   - **Never `gold_test`.** It is sealed and doesn't leave the laptop.
3. *(Optional, saves ~10 GB of downloading every session)* Add Input → Models → search **Qwen2.5**. If a `14b-instruct-awq` variation exists, attach it; cell 3 detects it on its own.
4. Run cells 1–2, then **Restart kernel** (vLLM swaps the torch version), then run from cell 3.

**Decision rule:** B2 has a 3 h box for 10k emails → needs **≥ 0.93 emails/s** at n=3. Cell 8 prints GO / SWITCH-TO-7B.

In [ ]:
# 1 — GPU check (shell only; don't import torch before the install)
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv
!python -V

In [ ]:
# 2 — Install. vLLM 0.8.5.post1 still has the V0 engine, which is the reliable path on Turing (T4, sm_75).
# After this cell: Kernel → Restart, then continue at cell 3.
!pip install -q "vllm==0.8.5.post1" "huggingface_hub[hf_transfer]" pyarrow openpyxl
!pip list 2>/dev/null | grep -Ei "^(vllm|torch|transformers|xgrammar|outlines) "

In [ ]:
# 3 — Config + model location
import os, glob, json, time, hashlib, random, datetime, pathlib
os.environ["VLLM_USE_V1"] = "0"                 # V0 engine on T4
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
MODEL_ID = "Qwen/Qwen2.5-14B-Instruct-AWQ"      # fallback: "Qwen/Qwen2.5-7B-Instruct-AWQ"
INPUT_ROOT = "/kaggle/input"
MAX_EMAIL_TOKENS = 1500     # teacher-side cap on the email body (student truncates at 384 separately)
MAX_MODEL_LEN = 8192
N_THROUGHPUT = 60           # gold-dev emails used for the speed test
RATIONALE = False           # True = model writes a 1-line reason before labels (slower; B1 can A/B this)

# Column names in the gold files. Adjust if finalize_gold.py used different ones (cell 4 prints the columns).
COL = dict(uid="uid", text="text", role="recipient_role", position="recipient_position", date="date")

random.seed(SEED)
RUN_DIR = pathlib.Path("/kaggle/working/runs") / f"{datetime.date.today():%Y-%m-%d}-teacher-throughput"
RUN_DIR.mkdir(parents=True, exist_ok=True)

def find_attached_model(model_id):
    tag = model_id.split("/")[-1].lower().replace("qwen2.5-", "")       # e.g. "14b-instruct-awq"
    for cfg in glob.glob(f"{INPUT_ROOT}/**/config.json", recursive=True):
        p = cfg.lower()
        if "qwen" in p and tag.replace("-", "") in p.replace("-", "").replace("_", ""):
            return os.path.dirname(cfg)
    return None

MODEL_PATH = find_attached_model(MODEL_ID)
if MODEL_PATH:
    print("Using attached Kaggle model:", MODEL_PATH)
else:
    from huggingface_hub import snapshot_download
    t0 = time.time()
    MODEL_PATH = snapshot_download(MODEL_ID, local_dir=f"/tmp/models/{MODEL_ID.split('/')[-1]}",
                                   allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.tiktoken", "merges.txt"])
    print(f"Downloaded {MODEL_ID} in {time.time()-t0:.0f}s ->", MODEL_PATH)
!du -sh {MODEL_PATH}

In [ ]:
# 4 — Inputs: find files, load gold-dev/calib, fingerprint everything (project rule: runs record input hashes)
import pandas as pd

def find_one(pattern):
    hits = sorted(glob.glob(f"{INPUT_ROOT}/**/{pattern}", recursive=True))
    assert hits, f"not found under {INPUT_ROOT}: {pattern}"
    return hits[0]

def load_table(path):
    return {".csv": pd.read_csv, ".parquet": pd.read_parquet, ".xlsx": pd.read_excel}[pathlib.Path(path).suffix](path)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

paths = {
    "gold_dev":   next(p for p in glob.glob(f"{INPUT_ROOT}/**/gold_dev.*",   recursive=True)),
    "gold_calib": next(p for p in glob.glob(f"{INPUT_ROOT}/**/gold_calib.*", recursive=True)),
    "guide":      find_one("labelling-guide.md"),
    "baseline":   find_one("human_baseline.json"),
    "text_py":    find_one("text.py"),
}
assert not glob.glob(f"{INPUT_ROOT}/**/gold_test*", recursive=True), "gold_test is attached. Remove it, it's sealed."
fingerprints = {k: sha256(v) for k, v in paths.items()}

dev, calib = load_table(paths["gold_dev"]), load_table(paths["gold_calib"])
GUIDE = open(paths["guide"], encoding="utf-8").read()
BASELINE = json.load(open(paths["baseline"]))
print("dev", dev.shape, "calib", calib.shape)
print("columns:", list(dev.columns))
missing = [c for c in COL.values() if c not in dev.columns]
print("MISSING columns (fix COL in cell 3):", missing) if missing else print("columns OK")
print("human baseline keys:", list(BASELINE)[:10])

In [ ]:
# 5 — Start vLLM: AWQ in fp16 (T4 has no bf16 and no Marlin), TP=2, prefix caching ON.
# Prefix caching is what makes this affordable: the guide + few-shots are one long shared prefix,
# computed once instead of once per email.
from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams
from transformers import AutoTokenizer

t0 = time.time()
llm = LLM(
    model=MODEL_PATH, quantization="awq", dtype="float16",
    tensor_parallel_size=2, disable_custom_all_reduce=True,   # T4 pair has no NVLink/P2P
    max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.92,
    enable_prefix_caching=True, seed=SEED, trust_remote_code=False,
)
tok = AutoTokenizer.from_pretrained(MODEL_PATH)
LOAD_S = time.time() - t0
print(f"engine up in {LOAD_S:.0f}s")
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv

In [ ]:
# 6 — Output schema + prompt builder (placeholder prompt; B1 replaces SYSTEM_RULES and FEW_SHOTS)
TYPES = ["Credentials", "Approval", "Meeting", "Task", "Info"]
URGENCY = ["Critical", "High", "Medium", "Low"]
SENTIMENT = ["Positive", "Neutral", "Negative", "Frustrated"]

props = {"type": {"enum": TYPES}, "urgency": {"enum": URGENCY}, "sentiment": {"enum": SENTIMENT}}
if RATIONALE:
    props = {"reason": {"type": "string", "maxLength": 200}, **props}
SCHEMA = {"type": "object", "properties": props, "required": list(props), "additionalProperties": False}

SYSTEM_RULES = (
    "You label corporate emails for a triage system. Follow the labelling guide exactly.\n"
    "Type precedence when several apply: Credentials > Approval/Meeting > Task > Info.\n"
    "Judge urgency as if today is the email's date.\n"
    "Answer with JSON only.\n\n# Labelling guide\n"
)
FEW_SHOTS = []   # B1: list of (user_msg, assistant_json) from gold-dev

def clip(text, n=MAX_EMAIL_TOKENS):
    ids = tok.encode(str(text), add_special_tokens=False)
    return text if len(ids) <= n else tok.decode(ids[:n]) + "\n[...truncated]"

def user_msg(row):
    return (f"Recipient role: {row[COL['role']]}\nRecipient position: {row[COL['position']]}\n"
            f"Email date (treat as today): {row[COL['date']]}\n\n--- EMAIL ---\n{clip(row[COL['text']])}")

def build_prompt(row):
    msgs = [{"role": "system", "content": SYSTEM_RULES + GUIDE}]   # identical for every email -> cached prefix
    for u, a in FEW_SHOTS:
        msgs += [{"role": "user", "content": u}, {"role": "assistant", "content": a}]
    msgs.append({"role": "user", "content": user_msg(row)})
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def sampling(n, temperature, max_tokens=None):
    return SamplingParams(n=n, temperature=temperature, top_p=0.95 if temperature > 0 else 1.0,
                          max_tokens=max_tokens or (160 if RATIONALE else 48), seed=SEED,
                          guided_decoding=GuidedDecodingParams(json=SCHEMA))

p0 = build_prompt(dev.iloc[0])
print("prompt tokens (email 0):", len(tok.encode(p0)), "| shared prefix tokens:",
      len(tok.encode(SYSTEM_RULES + GUIDE)))

In [ ]:
# 7 — Smoke test: 3 emails, greedy, check JSON validity
outs = llm.generate([build_prompt(r) for _, r in dev.head(3).iterrows()], sampling(1, 0.0))
for o in outs:
    txt = o.outputs[0].text
    print(json.loads(txt), "| finish:", o.outputs[0].finish_reason)

In [ ]:
# 8 — Throughput test: the real B2 setting (n=3, temp 0.7), cold cache vs warm
from collections import Counter

def run_batch(df):
    prompts = [build_prompt(r) for _, r in df.iterrows()]
    t0 = time.time()
    outs = llm.generate(prompts, sampling(3, 0.7))
    dt = time.time() - t0
    valid = total = unanimous = 0
    gen_tok = sum(len(c.token_ids) for o in outs for c in o.outputs)
    prm_tok = sum(len(o.prompt_token_ids) for o in outs)
    for o in outs:
        votes = []
        for c in o.outputs:
            total += 1
            try:
                votes.append(json.loads(c.text)); valid += 1
            except Exception:
                pass
        if len(votes) == 3 and len({v["type"] for v in votes}) == 1:
            unanimous += 1
    return dict(n_emails=len(df), seconds=round(dt, 1), emails_per_s=round(len(df) / dt, 3),
                json_valid_rate=round(valid / total, 4), type_unanimous_rate=round(unanimous / len(df), 3),
                prompt_tokens_per_email=round(prm_tok / len(df)), gen_tokens_per_email=round(gen_tok / len(df)))

sample = dev.sample(min(N_THROUGHPUT, len(dev)), random_state=SEED)
half = len(sample) // 2
cold = run_batch(sample.iloc[:half])     # first call also fills the prefix cache
warm = run_batch(sample.iloc[half:])     # steady state = what B2 will see
print("cold:", cold); print("warm:", warm)

rate = warm["emails_per_s"]
hours_10k = 10_000 / rate / 3600
NEED = 10_000 / (3 * 3600)
verdict = "GO (keep 14B)" if rate >= NEED else "SWITCH-TO-7B"
if "7B" in MODEL_ID and rate < NEED:
    verdict = "7B also too slow: cut pool or n, log it"
print(f"\n{MODEL_ID}: {rate:.2f} emails/s -> 10k in {hours_10k:.1f} h (box 3 h, need >= {NEED:.2f}/s) => {verdict}")

In [ ]:
# 9 — Log the run (project rule: runs/<date>-<name>/ with config, metrics, input fingerprints)
import vllm, torch, platform
config = dict(model_id=MODEL_ID, model_path=MODEL_PATH, quantization="awq", dtype="float16",
              tensor_parallel_size=2, max_model_len=MAX_MODEL_LEN, prefix_caching=True,
              max_email_tokens=MAX_EMAIL_TOKENS, rationale=RATIONALE, n=3, temperature=0.7, top_p=0.95,
              seed=SEED, n_throughput=len(sample), columns=COL,
              versions=dict(vllm=vllm.__version__, torch=torch.__version__, python=platform.python_version()))
metrics = dict(engine_load_s=round(LOAD_S), cold=cold, warm=warm, verdict=verdict,
               projected_hours_10k=round(hours_10k, 2))
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))
(RUN_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
(RUN_DIR / "inputs.sha256.json").write_text(json.dumps(fingerprints, indent=2))
print("saved ->", RUN_DIR); print(json.dumps(metrics, indent=2))

### If cell 8 says SWITCH-TO-7B
Set `MODEL_ID = "Qwen/Qwen2.5-7B-Instruct-AWQ"` in cell 3, **restart the kernel** (vLLM won't release the GPUs any other way), run 3→9 again, and add one line to `decision_log.md`: *"Teacher = Qwen2.5-7B-AWQ; 14B ran at X emails/s on 2×T4, below the 0.93/s needed for the B2 box."*

### Things to check before moving to B1
- `json_valid_rate` = 1.0 (guided decoding is on, so anything less is a bug, not noise)
- `prompt_tokens_per_email`: if > ~6k, lower `MAX_EMAIL_TOKENS` or shorten the guide
- Download `/kaggle/working/runs/…` and commit it to the ML repo